# Stage 5M & Stage 6: Closed-Loop Online Control & Multi-Model Benchmark
## Master Virtual GPU Execution Notebook (Kaggle Tesla P100 / Google Colab T4)

**Scientific Context & Authorizations:**
- **Stage 5M**: Closed-Loop Online Control authorized for controlled research evaluation under frozen protocol `v1.1.0-model-expansion` and ADR-005.
  - Operational Disparity Safety Gate (DSG) active during streaming.
  - Candidate updates evaluated strictly on the **disjoint Sentinel Panel ($N=30$ independent speakers, 300 clips)**.
  - Frozen safety thresholds: $\epsilon_R = 0.00\text{ pp}$, $\epsilon_G = +2.00\text{ pp}$, $\epsilon_D = +2.00\text{ pp}$, $B = 1000$, $\alpha = 0.05$.
  - Test-stream reference labels are quarantined via `LabelIsolationSanitizer` during adaptation.
  - Diagnostic comparison between operational sentinel decisions and retrospective stream outcomes.
- **Stage 6**: Six-Model Generalization Benchmark on Common Voice 27.0 ($K=4$, 225 windows, 900 clips across 60 speakers).
- **Stage 6.1**: Two-Model Large CTC Extension (`wav2vec2_large_lv60` and `wav2vec2_large_robust`).

**Instructions:**
1. In Kaggle, open the right-hand panel under **Notebook options**:
   - Set **Accelerator** to **GPU P100**.
   - Set **Internet** to **ON**.
2. Run all cells sequentially from top to bottom.

In [ ]:
# Step 1: Verify Hardware Accelerator (Tesla P100 / T4)
import torch
print("=== HARDWARE ACCELERATOR CHECK ===")
print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Device     : {gpu_name}")
    print(f"Total VRAM     : {vram_gb:.2f} GB")
    if "P100" in gpu_name or "T4" in gpu_name or vram_gb >= 14.0:
        print("Assessment     : EXCELLENT (High-performance 16GB GPU ready)")
else:
    print("WARNING: No GPU detected! Please enable GPU in Kaggle settings (Accelerator -> GPU P100).")

In [ ]:
# Step 2: Install dependencies, unpack audio assets & run preflight checks
!python cloud/kaggle/setup_kaggle.py

In [ ]:
# Step 3: Run Fast Preflight Smoke Test for Stage 5M (Verifies Sentinel Gate in ~30 seconds)
!python cloud/kaggle/run_stage5m_online_control.py --smoke-test

In [ ]:
# Step 4: Execute Full Stage 5M Closed-Loop Online Control Suite
# Evaluates CTC models (wav2vec2_base, data2vec_base, wav2vec2_100h) under acoustic stress
# with operational DSG control and Seq2Seq static controls.
!python cloud/kaggle/run_stage5m_online_control.py

In [ ]:
# Step 5: Display Stage 5M Operational Gate & Agreement Audit Results
import os
import pandas as pd

res_path = "results/stage5m/stage5m_closed_loop_results.csv"
agree_path = "results/stage5m/stage5m_decision_agreement_matrix.csv"
report_path = "reports/stage5m/stage5m_closed_loop_report.md"

if os.path.exists(res_path):
    print("=== STAGE 5M CLOSED-LOOP RECOGNITION OUTCOMES ===")
    df_res = pd.read_csv(res_path)
    display(df_res[["model_key", "condition", "method", "ordering_id", "final_wer", "final_disparity", "acceptance_rate", "false_approvals"]].head(25))

if os.path.exists(agree_path):
    print("\n=== STAGE 5M OPERATIONAL VS RETROSPECTIVE AGREEMENT MATRIX ===")
    df_agree = pd.read_csv(agree_path, index_col=0)
    display(df_agree)

if os.path.exists(report_path):
    print(f"\nFull Stage 5M Report written to: {report_path}")

In [ ]:
# Step 6: Execute Stage 6 Master Six-Model DSG Benchmark on Common Voice 27.0
# Evaluates wav2vec2_base, whisper_base, hubert_large, data2vec_base, distil_whisper_small, xlsr_english
# across 225 prequential windows with GPU memory purging between models.
!python cloud/kaggle/run_six_model_benchmark.py

In [ ]:
# Step 7: Execute Stage 6.1 Two-Model Large CTC Extension
# Evaluates wav2vec2-large-960h-lv60 (315.5M) and wav2vec2-large-robust (315.5M)
!python cloud/kaggle/run_stage6_1_extension.py

In [ ]:
# Step 8: Display Stage 6 Benchmark Master Tables
s6_bench = "reports/stage6/six_model_benchmark.csv"
s6_dsg = "reports/stage6/six_model_dsg_summary.csv"

if os.path.exists(s6_bench):
    print("=== MASTER SIX-MODEL BENCHMARK TABLE ===")
    df_bench = pd.read_csv(s6_bench)
    display(df_bench)

if os.path.exists(s6_dsg):
    print("\n=== DSG CONTROLLER SUMMARY TABLE ===")
    df_dsg = pd.read_csv(s6_dsg)
    display(df_dsg)

In [ ]:
# Step 9: Package Unified Results Bundle for Download
!python cloud/kaggle/export_results.py

In [ ]:
# Step 10: Verify Ready-to-Download Results Archive
bundle_path = "/kaggle/working/stage5_stage6_results_bundle.zip"
if not os.path.exists(bundle_path):
    bundle_path = "reports/stage5_stage6_results_bundle.zip"

if os.path.exists(bundle_path):
    sz_kb = os.path.getsize(bundle_path) / 1024
    print(f"SUCCESS: Unified results bundle ready for download ({sz_kb:.1f} KB): {bundle_path}")
    print("\n--> In Kaggle: Look in the right-hand panel under 'Output' -> Click the three dots next to 'stage5_stage6_results_bundle.zip' -> Download.")
else:
    print("Archive not found. Please ensure export step completed.")